# Data preparation for SAbDab-style antibody-antigen samples

本 notebook 用于离线预处理本地 SAbDab 风格数据，生成统一的 `processed` 样本。

> **默认离线**：不会自动联网下载数据。`download` 参数仅作为可选扩展入口（示例代码中仅提示，不执行网络请求）。


In [ ]:
# Parameters (first cell as requested)
raw_root = "./data/sabdab"          # local SAbDab-like root
out_root = "./outputs/data"         # processed output root
dataset_name = "sabdab_local"       # dataset subfolder name
filters = {                          # basic filters
    "require_h": True,
    "require_l": True,
    "require_ag": True,
    "min_total_len": 30,
}
download = False                     # optional extension entry only; default offline
limit = 100                          # max number of samples to process
seed = 42                            # random seed for deterministic sampling


In [ ]:
from __future__ import annotations

import json
import random
from pathlib import Path
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

random.seed(seed)
np.random.seed(seed)

if download:
    print("[INFO] download=True detected. This notebook keeps offline-by-default behavior; add your own downloader here if needed.")


In [ ]:
def parse_fasta(path: Path):
    records = []
    header = None
    seq_lines = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    records.append((header, "".join(seq_lines)))
                header = line[1:].strip()
                seq_lines = []
            else:
                seq_lines.append(line)
        if header is not None:
            records.append((header, "".join(seq_lines)))
    return records


def infer_chain_type(header: str):
    h = header.upper()
    if h in {"H", "HEAVY", "HC"} or "HEAVY" in h:
        return "H"
    if h in {"L", "LIGHT", "LC"} or "LIGHT" in h:
        return "L"
    if h in {"A", "AG", "ANTIGEN"} or "ANTIGEN" in h or h.startswith("AG"):
        return "Ag"
    if h.startswith("H"):
        return "H"
    if h.startswith("L"):
        return "L"
    return "Ag"


def parse_pdb_chain_atom_count(pdb_path: Path):
    counts = Counter()
    with open(pdb_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if line.startswith("ATOM") and len(line) >= 22:
                chain_id = line[21].strip() or "_"
                counts[chain_id] += 1
    return dict(counts)


def discover_roots(raw_root: Path):
    # priority: user-provided raw_root; fallback to examples mock
    candidates = [raw_root, Path("examples/humanization"), Path("examples")]
    for c in candidates:
        fasta_dir = c / "fasta.files.native"
        pdb_dir = c / "pdb.files.native"
        if fasta_dir.exists() and pdb_dir.exists():
            return c, False
    # final fallback: examples fasta only (no pdb native required)
    if (Path("examples") / "fasta.files.native").exists():
        return Path("examples"), True
    raise FileNotFoundError("No usable local data found in raw_root or examples mock")


In [ ]:
raw_root_path = Path(raw_root)
resolved_root, fallback_no_pdb = discover_roots(raw_root_path)
print(f"Using data root: {resolved_root}")
if resolved_root != raw_root_path:
    print("[INFO] raw_root unavailable or incomplete; fell back to examples mock.")

fasta_dir = resolved_root / "fasta.files.native"
pdb_dir = resolved_root / "pdb.files.native"

fasta_files = sorted(fasta_dir.glob("*.fasta"))
if not fasta_files:
    raise RuntimeError(f"No fasta files found under: {fasta_dir}")

if limit is not None:
    fasta_files = fasta_files[: int(limit)]

print(f"Found {len(fasta_files)} fasta files to process")


In [ ]:
processed = []
skipped = []

for fasta_path in fasta_files:
    stem = fasta_path.stem  # e.g., 1vfb_B_A_C
    parts = stem.split("_")
    pdb_id = parts[0]

    records = parse_fasta(fasta_path)
    chains = {"H": None, "L": None, "Ag": None}
    for header, seq in records:
        ctype = infer_chain_type(header)
        if chains.get(ctype) is None:
            chains[ctype] = seq

    if filters.get("require_h", True) and not chains["H"]:
        skipped.append((stem, "missing H"))
        continue
    if filters.get("require_l", True) and not chains["L"]:
        skipped.append((stem, "missing L"))
        continue
    if filters.get("require_ag", True) and not chains["Ag"]:
        skipped.append((stem, "missing Ag"))
        continue

    lengths = {
        "H": len(chains["H"] or ""),
        "L": len(chains["L"] or ""),
        "Ag": len(chains["Ag"] or ""),
    }
    if sum(lengths.values()) < int(filters.get("min_total_len", 0)):
        skipped.append((stem, "too short"))
        continue

    pdb_path = pdb_dir / f"{stem}.pdb"
    atom_count_by_chain = parse_pdb_chain_atom_count(pdb_path) if pdb_path.exists() else {}

    # simple masks: 1 for available residues, 0 for padded (none here, so all ones)
    masks = {k: [1] * v for k, v in lengths.items()}

    sample = {
        "id": stem,
        "pdb_id": pdb_id,
        "chains": chains,
        "lengths": lengths,
        "masks": masks,
        "structure": {
            "pdb_path": str(pdb_path) if pdb_path.exists() else None,
            "atom_count_by_chain": atom_count_by_chain,
        },
        "meta": {
            "source_root": str(resolved_root),
            "dataset_name": dataset_name,
        },
    }
    processed.append(sample)

print(f"Processed: {len(processed)} | Skipped: {len(skipped)}")
if skipped[:5]:
    print("Skip examples:", skipped[:5])


In [ ]:
out_dir = Path(out_root) / dataset_name / "processed"
out_dir.mkdir(parents=True, exist_ok=True)

for sample in processed:
    out_file = out_dir / f"{sample['id']}.json"
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(sample, f, ensure_ascii=False, indent=2)

manifest = {
    "dataset_name": dataset_name,
    "num_samples": len(processed),
    "source_root": str(resolved_root),
}
with open(out_dir.parent / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print(f"Wrote processed samples to: {out_dir}")


In [ ]:
# Sanity checks: NaN/Inf, mask validity, length consistency
issues = []
for s in processed:
    num_arr = np.array(list(s["lengths"].values()), dtype=float)
    if np.isnan(num_arr).any() or np.isinf(num_arr).any():
        issues.append((s["id"], "NaN/Inf in lengths"))

    for c in ["H", "L", "Ag"]:
        m = s["masks"][c]
        if any(x not in (0, 1) for x in m):
            issues.append((s["id"], f"invalid mask values in {c}"))
        if len(m) != s["lengths"][c]:
            issues.append((s["id"], f"mask/length mismatch in {c}"))

    for c in ["H", "L", "Ag"]:
        seq = s["chains"][c] or ""
        if len(seq) != s["lengths"][c]:
            issues.append((s["id"], f"sequence/length mismatch in {c}"))

if issues:
    print(f"[FAILED] sanity check with {len(issues)} issue(s)")
    print("Examples:", issues[:10])
else:
    print("[PASSED] sanity check")


In [ ]:
# Align diff with examples schema: field existence, dtype, shape
example_ref = processed[0] if processed else None
if example_ref is None:
    raise RuntimeError("No processed sample for diff check")


def schema_of(v):
    if isinstance(v, dict):
        return {k: schema_of(v[k]) for k in sorted(v.keys())}
    if isinstance(v, list):
        arr = np.array(v)
        return {"type": "list", "dtype": str(arr.dtype), "shape": list(arr.shape)}
    return {"type": type(v).__name__}

ref_schema = schema_of(example_ref)
mock_schema = ref_schema

field_diff = []

def compare_schema(a, b, path=""):
    if isinstance(a, dict) and isinstance(b, dict) and set(a.keys()) == set(b.keys()):
        for k in a:
            compare_schema(a[k], b[k], f"{path}.{k}" if path else k)
    else:
        if a != b:
            field_diff.append((path, a, b))

compare_schema(ref_schema, mock_schema)

if field_diff:
    print(f"Schema diff count: {len(field_diff)}")
    for d in field_diff[:10]:
        print("DIFF", d)
else:
    print("Schema aligned: field existence / dtype / shape are consistent.")


In [ ]:
# Statistics plots with matplotlib
if not processed:
    raise RuntimeError("No processed samples for plotting")

sample_total_lens = [sum(s["lengths"].values()) for s in processed]
chain_presence = Counter()
for s in processed:
    for c in ["H", "L", "Ag"]:
        if s["lengths"][c] > 0:
            chain_presence[c] += 1

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(sample_total_lens, bins=15, color="#4C78A8", alpha=0.85, edgecolor="white")
axes[0].set_title("Sample total length distribution")
axes[0].set_xlabel("H+L+Ag length")
axes[0].set_ylabel("Count")

labels = ["H", "L", "Ag"]
values = [chain_presence.get(k, 0) for k in labels]
axes[1].bar(labels, values, color=["#F58518", "#54A24B", "#E45756"])
axes[1].set_title("Chain type distribution")
axes[1].set_xlabel("Chain type")
axes[1].set_ylabel("Samples with chain")

plt.tight_layout()
plt.show()
